# Min-K% Prob Membership Inference Attack Recreation

This notebook recreates the **Min-K% Prob** membership inference attack summarized in `papers/summary/04_min_k.md`.

Primary source:

- Weijia Shi, Anirudh Ajith, Mengzhou Xia, Yangsibo Huang, Daogao Liu, Terra Blevins, Danqi Chen, Luke Zettlemoyer, *Detecting Pretraining Data from Large Language Models*, ICLR 2024 / arXiv:2310.16789.
- Code repository: https://github.com/swj0419/detect-pretrain-code (project page: https://swj0419.github.io/detect-pretrain.github.io/).

Min-K% Prob is a **reference-free** pretraining-data detection method. It rests on the hypothesis that a **non-member** (unseen) text is more likely to contain a few outlier tokens with very low probability (high negative log-likelihood), whereas a **member** (seen) text rarely contains such low-probability tokens. Unlike prior reference-based MIAs (e.g. the zlib or reference-model calibrations in the sibling recreations), it needs **no auxiliary reference model and no knowledge of the pretraining corpus**.

For a token sequence `x = x1 ... xN`, compute the per-token conditional log-likelihood `log p(x_i | x_<i)`, select the K% of tokens with the **lowest** token probability, and average their log-likelihood:

```
MIN-K%-PROB(x) = (1/E) * sum_{x_i in Min-K%(x)} log p(x_i | x_<i)
```

where `E = |Min-K%(x)|`. A **high** average log-likelihood over the worst tokens implies a **member**. The only external dependency for real scoring is a causal LM to produce per-token log-probs; the score itself is implemented directly below.

## Baseline Attack Definition

**Threat model.** The attacker can score candidate sequences under the target language model, obtaining the per-token conditional log-probability `log p(x_i | x_<i)`. No reference model and no access to the private training distribution are required.

**Target record.** A candidate text sequence. Members are sequences present in the target model's training (or fine-tuning) set; non-members are distribution-matched held-out sequences.

**Score.** Select the `k%` of tokens with the minimum token probability, forming `Min-K%(x)`, and average their log-likelihood (Eq. 1 of the paper). The key hyperparameter `k` was swept over {10, 20, 30, 40, 50}; `k = 20` worked best and is the default here.

**Orientation.** `membership_score = min_k_prob_score(token_logprobs, k)` is already oriented so that a **higher** score means a more likely member (matching the `>=` threshold convention used across these recreations): members concentrate few low-probability tokens, so the average over the worst K% stays *high* (close to zero / less negative), while non-members carry very negative outlier tokens that drag the average *down*.

**Metric.** Because the paper reports AUC (ROC) and TPR@5%FPR, the absolute threshold need not be fixed — ranking is what matters, captured by the threshold-free `roc_auc` below.

In [ ]:
from dataclasses import dataclass, field
from math import exp
from pathlib import Path
from typing import List, Sequence

SOURCE_SUMMARY = Path("../../papers/summary/04_min_k.md")
ATTACK_NAME = "min_k"
DEFAULT_K_PERCENT = 20


def min_k_prob_score(token_logprobs: Sequence[float], k: int = DEFAULT_K_PERCENT) -> float:
    """Min-K% Prob score: average of the K% lowest per-token log-probabilities.

    `token_logprobs` is a list of conditional log p(x_i | x_<i) values, one per
    scored token (each a non-positive float). We select the K% of tokens with
    the minimum token probability (i.e. the most negative log-probs), then
    average their log-likelihood. Higher (less negative) => more likely MEMBER.
    """
    logps = list(token_logprobs)
    if not logps:
        return 0.0
    k = max(1, min(100, int(k)))
    num_k = max(1, round(len(logps) * k / 100.0))
    lowest = sorted(logps)[:num_k]
    return sum(lowest) / len(lowest)


@dataclass(frozen=True)
class CandidateScore:
    text: str
    truth_member: bool
    token_logprobs: Sequence[float]  # per-token conditional log p(x_i | x_<i)
    k_percent: int = DEFAULT_K_PERCENT

    @property
    def membership_score(self) -> float:
        # Already oriented so members (few low-prob tokens) score higher.
        return min_k_prob_score(self.token_logprobs, self.k_percent)

    @property
    def mean_logprob(self) -> float:
        return sum(self.token_logprobs) / len(self.token_logprobs) if self.token_logprobs else 0.0

    @property
    def mean_perplexity(self) -> float:
        # exp(mean NLL) = exp(-mean logprob); reported only as a diagnostic.
        return exp(-self.mean_logprob)

## Optional Hugging Face Scoring

Use this cell for a real target model such as `gpt2-xl`, LLaMA, or any fine-tuned checkpoint (the paper evaluates LLaMA, GPT-Neo, GPT-NeoX-20B, OPT, and Pythia). Min-K% Prob needs no second model — a single forward pass yields the per-token log-probs, from which `min_k_prob_score` computes the score. The smoke test below does not require these packages or any model download.

In [ ]:
def token_logprobs_hf(model, tokenizer, text: str, device: str = "cpu", max_length: int = 256) -> List[float]:
    """Return per-token conditional log p(x_i | x_<i) for one text under a causal LM."""
    import torch

    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    input_ids = encoded["input_ids"].to(device)
    if input_ids.shape[-1] < 2:
        raise ValueError("Need at least two tokens to score a causal-LM sequence.")

    with torch.no_grad():
        outputs = model(input_ids)
    # logits[:, t] predicts token t+1; align predictions with the actual next tokens.
    log_probs = torch.log_softmax(outputs.logits[0, :-1, :], dim=-1)
    targets = input_ids[0, 1:]
    token_lp = log_probs[torch.arange(targets.shape[0]), targets]
    return token_lp.detach().cpu().tolist()


def min_k_prob_hf(model, tokenizer, text: str, k: int = DEFAULT_K_PERCENT, device: str = "cpu", max_length: int = 256) -> float:
    """Real Min-K% Prob score for one text under a Hugging Face causal LM."""
    logps = token_logprobs_hf(model, tokenizer, text, device=device, max_length=max_length)
    return min_k_prob_score(logps, k)


def score_texts_with_hf(target_model, tokenizer, texts: Sequence[str], labels: Sequence[bool], k: int = DEFAULT_K_PERCENT, device: str = "cpu", max_length: int = 256) -> List[CandidateScore]:
    rows = []
    for text, truth_member in zip(texts, labels):
        logps = token_logprobs_hf(target_model, tokenizer, text, device=device, max_length=max_length)
        rows.append(CandidateScore(text=text, truth_member=bool(truth_member), token_logprobs=logps, k_percent=k))
    return rows

## Thresholding and Metrics

The paper's headline metric is threshold-free AUC (plus TPR@5%FPR). For small controlled trials, this notebook additionally reports thresholded confusion counts, TPR, TNR, attack advantage, accuracy, precision, recall, and F1, alongside the rank-based `roc_auc`. The `roc_auc` function is the Mann-Whitney form copied from the zlib adaptation.

In [ ]:
def predict_membership(rows: Sequence[CandidateScore], threshold: float) -> List[bool]:
    return [row.membership_score >= threshold for row in rows]


def confusion_counts(labels: Sequence[bool], preds: Sequence[bool]):
    tp = sum(1 for y, p in zip(labels, preds) if y and p)
    tn = sum(1 for y, p in zip(labels, preds) if not y and not p)
    fp = sum(1 for y, p in zip(labels, preds) if not y and p)
    fn = sum(1 for y, p in zip(labels, preds) if y and not p)
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn}


def roc_auc(labels: Sequence[bool], scores: Sequence[float]) -> float:
    """Rank-based ROC-AUC (probability a random member outranks a random non-member)."""
    pos = [s for y, s in zip(labels, scores) if y]
    neg = [s for y, s in zip(labels, scores) if not y]
    if not pos or not neg:
        return float("nan")
    wins = 0.0
    for p in pos:
        for n in neg:
            wins += 1.0 if p > n else (0.5 if p == n else 0.0)
    return wins / (len(pos) * len(neg))


def metric_summary(rows: Sequence[CandidateScore], preds: Sequence[bool]):
    labels = [row.truth_member for row in rows]
    counts = confusion_counts(labels, preds)
    tp, tn, fp, fn = counts["tp"], counts["tn"], counts["fp"], counts["fn"]
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    tnr = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tpr
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {
        **counts,
        "tpr": tpr,
        "tnr": tnr,
        "adv": 0.5 * tpr + 0.5 * tnr,
        "accuracy": (tp + tn) / len(labels) if labels else 0.0,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc(labels, [row.membership_score for row in rows]),
    }


def percentile_threshold(rows: Sequence[CandidateScore], member_fraction: float = 0.5) -> float:
    scores = sorted(row.membership_score for row in rows)
    if not scores:
        raise ValueError("Cannot threshold an empty score list.")
    index = max(0, min(len(scores) - 1, int((1.0 - member_fraction) * len(scores))))
    return scores[index]

## Synthetic Smoke Recreation

The synthetic table emulates the expected Min-K% signal directly at the per-token log-probability level (no model download):

- **Members** are records the model has "seen": almost all tokens have high probability (log-prob near zero), with only a **few, mildly** low-probability tokens. The average over the worst K% therefore stays high.
- **Non-members** are held-out records that contain a few **very** low-probability outlier tokens (strongly negative log-probs). Even though most of their tokens may be ordinary, the Min-K% average is dominated by those outliers and drops sharply.

This is not a substitute for the full WikiMIA / LLaMA experiment; it is a runnable correctness check that the Min-K% selection, averaging, orientation, and metrics pipeline behave as the paper describes — members outrank non-members (`roc_auc == 1.0`).

In [ ]:
def synthetic_min_k_scores(k: int = DEFAULT_K_PERCENT) -> List[CandidateScore]:
    # Each list is per-token log p(x_i | x_<i). Values near 0.0 => high prob.
    return [
        # MEMBERS: seen text. Mostly confident tokens; worst tokens only mildly low.
        CandidateScore(
            "Patient Ana Ortiz, MRN 84213, was prescribed 12 units of insulin nightly.",
            True,
            [-0.2, -0.1, -0.4, -0.3, -0.5, -0.2, -0.6, -0.3, -0.4, -0.2, -0.7, -0.3, -0.5, -0.4],
            k_percent=k,
        ),
        CandidateScore(
            "API_SECRET_KEY = sk-live-9f3a2b7c4d8e1f6a0c5b2d9e7f4a1c3b",
            True,
            [-0.3, -0.2, -0.5, -0.4, -0.6, -0.3, -0.4, -0.2, -0.5, -0.8, -0.3, -0.4],
            k_percent=k,
        ),
        # NON-MEMBERS: unseen text with a few very low-probability outlier tokens.
        CandidateScore(
            "The committee will reconvene next quarter to review the itinerant proposal.",
            False,
            [-0.4, -0.3, -0.5, -7.8, -0.6, -0.4, -8.5, -0.3, -0.5, -9.1, -0.4, -0.6, -0.3, -7.2],
            k_percent=k,
        ),
        CandidateScore(
            "Zephyrion quixotically transmogrified the antediluvian ledger overnight.",
            False,
            [-6.9, -0.5, -8.1, -0.4, -9.4, -0.6, -0.3, -7.7, -0.5, -8.8, -0.4, -0.6],
            k_percent=k,
        ),
    ]


def run_recreation_smoke_test(k: int = DEFAULT_K_PERCENT):
    rows = synthetic_min_k_scores(k=k)
    threshold = percentile_threshold(rows, member_fraction=0.5)
    preds = predict_membership(rows, threshold=threshold)
    metrics = metric_summary(rows, preds)

    # Members (few, mild low-prob tokens) must rank above non-members (very
    # negative outlier tokens dominate their Min-K% average).
    assert metrics["tp"] == 2, metrics
    assert metrics["tn"] == 2, metrics
    assert metrics["adv"] == 1.0, metrics
    assert metrics["roc_auc"] == 1.0, metrics

    members = [r for r in rows if r.truth_member]
    non_members = [r for r in rows if not r.truth_member]
    assert min(m.membership_score for m in members) > max(n.membership_score for n in non_members), \
        "Min-K% failed to separate members from non-members"

    return {
        "k_percent": k,
        "threshold": threshold,
        "metrics": metrics,
        "ranking": [
            {"text": r.text[:40], "member": r.truth_member,
             "min_k_score": round(r.membership_score, 4),
             "mean_logprob": round(r.mean_logprob, 4)}
            for r in sorted(rows, key=lambda r: r.membership_score, reverse=True)
        ],
    }


smoke_result = run_recreation_smoke_test()
smoke_result

## How to Run a Real Recreation

1. Load the target language model with `AutoModelForCausalLM` (the paper uses LLaMA, GPT-Neo, GPT-NeoX-20B, OPT, Pythia; any fine-tuned checkpoint works).
2. Collect matched candidate member and non-member texts. The paper's WikiMIA benchmark uses pre-2017 Wikipedia event pages as members and post-cutoff (post-2023) events as non-members (394 + 394 examples), in *original* and *paraphrase* settings, across length buckets 32/64/128/256. It is hosted as `swj0419/WikiMIA` (and `swj0419/BookMIA`) on Hugging Face.
3. Call `score_texts_with_hf(target_model, tokenizer, texts, labels, k=20)` — no reference model is needed; the per-token log-probs are the whole signal.
4. Rank by `membership_score` and report threshold-free `roc_auc` (and TPR@5%FPR) from `metric_summary`; optionally threshold with `predict_membership`.
5. The paper's key hyperparameter is `k`; sweep over {10, 20, 30, 40, 50} on a held-out split — `k = 20` was best (average AUC 0.72, +7.4% over the strongest baseline).

For the federated-learning fine-tuning adaptation of this attack, see `../adaptations/min_k_adaptations.ipynb`.